In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import random


from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    balanced_accuracy_score,
    recall_score,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt

# ==============================
# CONFIG
# ==============================

SEED = 42
np.random.seed(SEED)
random.seed(SEED)


TARGET_COL = "label"   # hedef sütun adını değiştir

THRESHOLDS = {
    "A": {
        "lightgbm": 0.6263,
        "xgboost": 0.3333,
        "logistic regression":  0.7374
    },
    "B": {
        "lightgbm": 0.2828,
        "xgboost": 0.4444,
        "logistic regression": 0.5758
    },
    "C": {
        "lightgbm": 0.3333,
        "xgboost": 0.7879,
        "logistic regression": 0.5960
    }
}

RESULTS_DIR = "results/pah"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ==============================
# METRİK HESAPLAMA
# ==============================

def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)

    roc_auc = roc_auc_score(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred, average='macro')
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    sensitivity = recall_score(y_true, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)

    return {
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "f1": f1,
        "balanced_accuracy": bal_acc,
        "sensitivity": sensitivity,
        "specificity": specificity
    }

# ==============================
# MODEL TANIMLARI
# ==============================

def get_models(y_train):

  neg = np.sum(y_train == 0)
  pos = np.sum(y_train == 1)
  scale_pos_weight = neg / pos

  return {
      "lightgbm": lgb.LGBMClassifier(random_state=SEED, verbosity=-1),
      "xgboost": xgb.XGBClassifier(random_state=SEED, eval_metric='logloss'),
      "logistic regression": LogisticRegression(random_state=SEED, max_iter=1000, class_weight="balanced", solver="saga", n_jobs=-1)
    }

# ==============================
# SHAP + FEATURE IMPORTANCE
# ==============================

def save_feature_importance(model, model_name, X, save_path):
    if hasattr(model, "feature_importances_"):
        importance = model.feature_importances_
    else:
        return

    df_imp = pd.DataFrame({
        "feature": X.columns,
        "importance": importance
    }).sort_values(by="importance", ascending=False)

    df_imp.to_csv(os.path.join(save_path, "feature_importance.csv"), index=False)


def save_group_shap(shap_values, X, group_dict, save_path):

    import matplotlib.pyplot as plt

    group_importance = {}

    for group_name, features in group_dict.items():
        idx = [X.columns.get_loc(f) for f in features if f in X.columns]

        if len(idx) == 0:
            continue

        group_vals = np.abs(shap_values[:, idx])

        group_importance[group_name] = {
            "mean": group_vals.mean(),
            "std": group_vals.std(),
            "num_features": len(idx)
        }

    df = pd.DataFrame([
        {
            "group": g,
            "mean_importance": v["mean"],
            "std_importance": v["std"],
            "num_features": v["num_features"]
        }
        for g, v in group_importance.items()
    ])

    # normalize (çok önemli)
    df["normalized_importance"] = df["mean_importance"] / df["mean_importance"].sum()

    df = df.sort_values(by="mean_importance", ascending=False)

    # ======================
    # CSV KAYDET
    # ======================
    df.to_csv(os.path.join(save_path, "group_shap_importance.csv"), index=False)

    # ======================
    # PNG KAYDET
    # ======================
    plt.figure()
    plt.barh(df["group"], df["normalized_importance"])
    plt.gca().invert_yaxis()
    plt.xlabel("Normalized Importance")
    plt.title("Group SHAP Importance")

    plt.tight_layout()
    plt.savefig(os.path.join(save_path, "group_shap_importance.png"))
    plt.close()

def save_shap(model, X, save_path):
    # Logistic Regression mı kontrol et
    if isinstance(model, LogisticRegression):
        explainer = shap.LinearExplainer(model, X, feature_perturbation="interventional")
        shap_values = explainer.shap_values(X)
    else:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X)

    plt.figure()
    shap.summary_plot(shap_values, X, show=False)
    plt.savefig(os.path.join(save_path, "shap_summary.png"))
    plt.close()

    save_group_shap(shap_values, X, group_dict, save_path)

# ==============================
# ANA PIPELINE
# ==============================

def run_experiment(train_df, test_df, feature_list, group_name, y_train):
    print(f"\n=== {group_name} başlıyor ===")

    group_path = os.path.join(RESULTS_DIR, group_name)
    os.makedirs(group_path, exist_ok=True)

    X_train = train_df[feature_list]
    y_train = train_df[TARGET_COL]

    X_test = test_df[feature_list]
    y_test = test_df[TARGET_COL]

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    models = get_models(y_train)

    for model_name, model in models.items():
        print(f"{model_name} eğitiliyor...")

        model_path = os.path.join(group_path, model_name)
        os.makedirs(model_path, exist_ok=True)

        threshold = THRESHOLDS[group_name][model_name]

        # Train
        model.fit(X_train_scaled, y_train)

        # Predict
        y_prob = model.predict_proba(X_test_scaled)[:, 1]

        # Metrics
        metrics = compute_metrics(y_test, y_prob, threshold)

        # Save metrics
        with open(os.path.join(model_path, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=4)

        # Save model
        joblib.dump(model, os.path.join(model_path, "model.pkl"))

        # Feature importance
        save_feature_importance(model, model_name, X_train, model_path)

        # SHAP
        save_shap(model, pd.DataFrame(X_test_scaled, columns=feature_list), model_path)

        print(f"{model_name} tamamlandı.")

# ==============================
# KULLANIM
# ==============================

# SEN BURAYI DOLDURACAKSIN

# örnek:
train_df = pd.read_csv("pah_train.csv")
test_df = pd.read_csv("pah_test.csv")

# feature listeleri
# features_a = [...]
# features_b = [...]
# features_c = [...]

#Meta predictors (Ensemble modeller)
pure_meta_predictors = [
    #classic
    'REVEL_score_count',
    'REVEL_score_max',
    'REVEL_score_mean',
    'REVEL_score_std',
    'MetaLR_score',
    'MetaLR_score_was_na',
    'MetaSVM_score',
    'MetaSVM_score_was_na',
    'BayesDel_addAF_score',
    'BayesDel_addAF_score_was_na',
    'BayesDel_noAF_score',
    'BayesDel_noAF_score_was_na',
    'M-CAP_score',
    'M-CAP_score_was_na',
    'Eigen-phred_coding',
    'Eigen-phred_coding_was_na',
    'Eigen-raw_coding',
    'Eigen-raw_coding_was_na',
    'fathmm-XF_coding_score',
    'fathmm-XF_coding_score_was_na',
    'gMVP_score_count',
    'gMVP_score_max',
    'gMVP_score_mean',
    'gMVP_score_std',
    'CADD_phred']

new_dl_meta_like = [
    #new
    'AlphaMissense_rankscore',
    'AlphaMissense_rankscore_was_na',
    'AlphaMissense_score_count',
    'AlphaMissense_score_max',
    'AlphaMissense_score_mean',
    'AlphaMissense_score_std',
    'ESM1b_score_count',
    'ESM1b_score_max',
    'ESM1b_score_mean',
    'ESM1b_score_std',
    'MutPred2_score_count',
    'MutPred2_score_max',
    'MutPred2_score_mean',
    'MutPred2_score_std',
    'popEVE_score_count',
    'popEVE_score_max',
    'popEVE_score_mean',
    'popEVE_score_std',
    'DANN_score',
    'DANN_score_was_na'
]

#Fonksiyonel predictors (Tekil modeller)
functional_predictors = [
    'SIFT_score_count',
    'SIFT_score_max',
    'SIFT_score_mean',
    'SIFT_score_std',
    'MutationAssessor_score_count',
    'MutationAssessor_score_max',
    'MutationAssessor_score_mean',
    'MutationAssessor_score_std',
    'MutationTaster_score_count',
    'MutationTaster_score_max',
    'MutationTaster_score_mean',
    'MutationTaster_score_std',
    'Polyphen2_HDIV_score_count',
    'Polyphen2_HDIV_score_max',
    'Polyphen2_HDIV_score_mean',
    'Polyphen2_HDIV_score_std',
    'VEST4_score_count',
    'VEST4_score_max',
    'VEST4_score_mean',
    'VEST4_score_std',
    'MisFit_D_score_count',
    'MisFit_D_score_max',
    'MisFit_D_score_mean',
    'MisFit_D_score_std'
]

#Evrimsel korunmuşluk
conservation_scores = [
    #gerp tabanlı
    'GERP++_RS',
    'GERP++_RS_was_na',
    'GERP_92_mammals',
    'GERP_92_mammals_was_na',
    #olasılık tabanlı
    'phastCons100way_vertebrate',
    'phastCons470way_mammalian',
    #evrimsel hız tabanlı
    'phyloP100way_vertebrate',
    'phyloP470way_mammalian',
    'phyloP470way_mammalian_was_na',
]

#Popülasyon Frekansı
population_score = [
    'gnomAD4.1_joint_AF',
    'gnomAD4.1_joint_AF_was_na',
    'gnomAD4.1_joint_POPMAX_AF',
    'gnomAD4.1_joint_POPMAX_AF_was_na',
    'dbNSFP_POPMAX_AF',
    'dbNSFP_POPMAX_AF_was_na'
]

# MANE
mane = [
    'MANE_Empty',
    'MANE_Select'
]

A_full = pure_meta_predictors+new_dl_meta_like+functional_predictors+conservation_scores+population_score+mane
B_no_meta_predictors = new_dl_meta_like+conservation_scores+population_score+mane
C_pure = conservation_scores+population_score+mane

group_dict = {
    "meta_predictors": pure_meta_predictors,
    "dl_meta": new_dl_meta_like,
    "functional": functional_predictors,
    "conservation": conservation_scores,
    "population": population_score,
    "mane": mane
}

# ÇALIŞTIR

y_train = train_df["label"]
run_experiment(train_df, test_df, A_full, "A", y_train)
run_experiment(train_df, test_df, B_no_meta_predictors, "B", y_train)
run_experiment(train_df, test_df, C_pure, "C", y_train)